[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mauriciorslrv/DS_basics/blob/main/03%20Aprendizaje%20M%C3%A1quina/notebooks/04_Regresion_Demanda_Bicicletas.ipynb)

# 04 · Regresión: estimar demanda de bicicletas

> **Pregunta:** ¿qué tan bien podemos estimar los alquileres por hora usando calendario y condiciones reportadas?

Usamos el dataset real UCI Bike Sharing. El objetivo **cnt** cuenta las bicicletas alquiladas en cada hora. Excluimos **casual** y **registered**, porque son partes del total y provocarían fuga de información.

📊 [UCI Bike Sharing · CC BY 4.0](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) · DOI 10.24432/C5W894.

## 1. Cargar y explorar

In [ ]:
%pip -q install ucimlrepo

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo

bike = fetch_ucirepo(id=275)
df = bike.data.features.copy()
df["cnt"] = bike.data.targets.iloc[:, 0].to_numpy()
df["dteday"] = pd.to_datetime(df["dteday"])

print("Dimensiones:", df.shape)
print("Periodo:", df["dteday"].min().date(), "a", df["dteday"].max().date())
display(df.head())

### Patrón temporal

Agrupar por semana permite ver cambios a lo largo del tiempo. Esta vista ayuda a pensar si una partición al azar representaría el uso futuro.

In [ ]:
weekly = df.set_index("dteday")["cnt"].resample("W").mean()
weekly.plot(figsize=(11, 3), color="#176b87", title="Demanda media semanal")
plt.ylabel("Alquileres por hora")
plt.show()

## 2. Definir predictores sin filtrar la respuesta

Usaremos calendario y condiciones disponibles en la tabla. Quitamos el identificador y la fecha; la fecha se conserva sólo para ordenar. Para un pronóstico real, el clima observado podría no estar disponible: habría que usar un pronóstico meteorológico publicado antes de la hora estimada.

In [ ]:
target = "cnt"
excluded = [c for c in ["casual", "registered", "instant", "dteday"] if c in df.columns]
features = [c for c in df.columns if c not in [target, *excluded]]
print("Excluidas:", excluded)
print("Predictoras:", features)
print("Componentes suman cnt:",
      (df["casual"] + df["registered"] == df[target]).all())

## 3. Simular predicción hacia el futuro

Entrenamos con el primer 80 % de fechas y reservamos el bloque posterior. El test no participa en seleccionar ni ajustar el modelo.

In [ ]:
dates = np.sort(df["dteday"].unique())
cutoff = dates[int(0.80 * len(dates))]
train = df[df["dteday"] < cutoff].copy()
test = df[df["dteday"] >= cutoff].copy()
X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]
print("Fecha de corte:", pd.Timestamp(cutoff).date())
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 4. Baseline

La mediana de entrenamiento es una referencia simple. Si el modelo no mejora de forma útil sobre esta regla, su complejidad no aporta valor evidente.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)
pred_baseline = baseline.predict(X_test)
print("MAE baseline:", round(mean_absolute_error(y_test, pred_baseline), 2), "bicicletas")

## 5. Entrenar una regresión basada en árboles

Tratamos códigos de calendario como categorías. El preprocesamiento se ajusta dentro del pipeline y el bosque aprende relaciones no lineales entre calendario, clima y demanda. El modelo es un ejemplo para comparar, no una recomendación universal.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

categorical = [c for c in ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit"] if c in X_train]
numeric = [c for c in X_train if c not in categorical]

preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical)
])
model = Pipeline([
    ("prep", preprocess),
    ("regressor", RandomForestRegressor(
        n_estimators=120, min_samples_leaf=3, n_jobs=-1, random_state=42
    ))
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Modelo ajustado.")

## 6. Leer las métricas

**MAE** comunica el error absoluto promedio en bicicletas. **RMSE** penaliza más los errores grandes. **R²** compara con una referencia basada en la media y puede ser negativo.

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred)
    }

results = pd.DataFrame({
    "Baseline mediana": regression_metrics(y_test, pred_baseline),
    "Random Forest": regression_metrics(y_test, pred)
}).T
results.round(2)

### ¿Dónde se equivoca?

La línea diagonal indica predicción perfecta. El gráfico de error absoluto medio por hora muestra si el rendimiento cambia entre horas de baja y alta actividad.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, pred, alpha=0.2, s=12, color="#176b87")
limit = max(float(y_test.max()), float(np.max(pred)))
plt.plot([0, limit], [0, limit], "--", color="#b64a36")
plt.xlabel("Demanda observada")
plt.ylabel("Demanda estimada")
plt.title("Test temporal: observado vs. estimado")
plt.show()

errors = test[["hr"]].copy()
errors["abs_error"] = np.abs(y_test.to_numpy() - pred)
errors.groupby("hr")["abs_error"].mean().plot(figsize=(9, 3), marker=".", color="#b64a36")
plt.title("MAE por hora del día")
plt.xlabel("Hora")
plt.ylabel("MAE (bicicletas)")
plt.show()

## 🧠 Interpretación y límites

La prueba representa un bloque posterior de fechas de este sistema, no otra ciudad o futuro lejano. El clima histórico observado puede no estar disponible antes de la predicción. Cambios de estación, eventos, políticas o infraestructura pueden degradar el desempeño.

## 🧪 Mini reto

Prueba un segundo regresor y compara MAE y RMSE con el baseline. Describe una hora o periodo con error alto y plantea una explicación que se pueda verificar con más datos.

## 🚀 Siguiente

La notebook 05 cambia de una cantidad numérica a una etiqueta sí/no usando una campaña bancaria real.

## Fuentes y atribución

- Fanaee-T, H. & Gama, J. (2013). UCI Bike Sharing Dataset. [Ficha y licencia CC BY 4.0](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset), DOI 10.24432/C5W894.
- Scikit-learn: [métricas de regresión](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics) y [pipelines](https://scikit-learn.org/stable/modules/compose.html).